#### get text content from data

In [1]:
import pandas as pd
import re
import json
import ast
from os import fdopen, remove
from tempfile import mkstemp
from shutil import move, copymode
from pprint import pprint 
import os
import string
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def flatten(xss):
    return [x for xs in xss for x in xs]

In [2]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    return [token for token in token_list if not token.is_stop]

    
def lemmatizing(token_list):
    return [token.lemma_ for token in token_list]
    
def get_POS(token_list):
    return [token.pos_ for token in token_list]

In [3]:
# AI-terms from ANTHROSCORE paper, transalted with deepL
ai_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
regex_ai = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ai_terms])[:-1]

# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] + ai_terms
ki_terms = [x.lower() for x in ki_terms]
regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b|\bsprachmodelle\b|\bsprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_7029/3111368924.py:3: SyntaxWarning: invalid escape sequence '\s'
  regex_ai = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ai_terms])[:-1]
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_7029/3111368924.py:10: SyntaxWarning: invalid escape sequence '\s'
  regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]


In [4]:
# get sentences with ki_terms
from importlib.resources import path


sentences_file = r"deu_news_2024_10K/deu_news_2024_10K-sentences.txt"
def to_sentences(sentences_file):
    sentences = []
    with open(sentences_file) as f:
        for line in f:
            txt = line.split('\t')
            #split sentences
            txt_content = txt[1::]
            # filter ki_terms
            sentences.append(remove_punctuation(line.lower()))  
    return sentences


ki_sentences = {}
sentences = to_sentences(sentences_file)
for sent in sentences:       
    if len(sent) > 0 and re.search(regex_ki, sent):
        #print("res",sent, re.search(regex_ki, sent).string)
        split = re.sub(r"\t", "-", sent).split("-")
        ki_sentences[split[0]] = split[1]

print(ki_sentences)
print(len(ki_sentences))

with open(f'generated_data/Leipzig_KI_sentences.json', 'w') as f:
    json.dump(ki_sentences, f)

{'41': 'aber das was der maturand in dem erwähnten text in der nzz gemacht hat sollten sie ebenfalls mitbringen er hat sich dank ki in kürzester zeit viel wissen angeeignet\n', '153': 'ai vorzustellen ein beweis für unsere bemühungen der letzten jahre bei der einführung neuester technologien zur umgestaltung der kundenbeziehung führend zu sein\n', '235': 'allerdings wirft der einsatz der großen sprachmodelle large languag models llms rechtliche fragen auf\n', '721': 'auch sollen die akkus im ai pin selbst sowie in den battery boostern nicht betroffen sein\n', '761': 'auf dem mwc 2024 in barcelona kündigte das unternehmen diese woche den qualcomm ai hub an eine sammlung von mehr als 75 modellen die speziell für qualcomm und snapdragonplattformen optimiert wurden\n', '1234': 'bislang kann man openai nicht einholen obwohl etwa die und dalle 3 bereits funktionen bietet die bis dato der abovariante chatgpt plus vorbehalten waren\n', '1600': 'darüber hinaus haben künstliche intelligenz ki un

In [5]:
# save ki_sentences to file
with open(r"generated_data/Leipzig_KI_sentences.txt", "w") as f:
        f.write(str(ki_sentences))

In [6]:
sent_dict = {}
for sent in sentences:       
    split = re.sub(r"\t", "-", sent).split("-")
    sent_dict[split[0]] = split[1]


In [7]:
# get KI-terms in "...words.txt" file
words_file = r"deu_news_2024_10K/deu_news_2024_10K-words.txt"
# Convert words file to tab-separated for csv reading
    #Create temp file
fh, abs_path = mkstemp()
with fdopen(fh,'w') as new_file:
    with open(words_file) as old_file:
        for line in old_file:
            new_file.write(line.replace(' ',';').replace('\t',';').replace('"','\\"'))
#Copy the file permissions from the old file to the new file, then remove the old file and
copymode(words_file, abs_path)
remove(words_file)
move(abs_path, words_file)

# filter ki_terms
data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
words = data[1].tolist()

ki_words = [w.lower() for w in words if not type(w) == float and w.lower() in ki_terms]
pprint(ki_words)

['ki',
 'ai',
 'chatgpt',
 'gemini',
 'chatbot',
 'llms',
 'sprachmodell',
 'sprachmodelle']


### TF-IDF

In [8]:
# use tf-idf to get keywords from texts
# corpus a list of sentences/documents
# stopwords_l a list of stopwords in the target language if not english
def calc_tf_idf(corpus, stopwords_l= None):
    corpus_cleaned = [remove_punctuation(doc.lower()) for doc in corpus]
    if stopwords_l:
        corpus_cleaned = [remove_stop_words_from_text(doc.split()) for doc in corpus_cleaned]
        vectorizer = TfidfVectorizer(stop_words=stopwords_l, lowercase=True, ngram_range=(1,2))
    else:
        vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1,2))
    # fit tfidf
    tfidf_m = vectorizer.fit_transform(corpus_cleaned) #transform
    #print(tfidf_m.shape)
    feature_array = vectorizer.get_feature_names_out()
    # top (20?) for every doc
    tfidf_sorting = np.argsort(tfidf_m.toarray()).flatten()[::-1] 
    top_n = feature_array[tfidf_sorting].tolist()[:20]
    print(top_n)
    return top_n #array of list(s?)

In [9]:
tfidf_words = calc_tf_idf(ki_sentences.values(), stopwords)
with open(r"generated_data/Leipzig_KI_sentences_tfidf.txt", "w") as f:
        f.write(str(tfidf_words))

['sonderpreis ai', 'zusätzlich sonderpreis', 'zusätzlich', 'vergeben', 'ai communication', 'communication vergeben', 'sonderpreis', 'communication', 'ai', 'financial', 'feuer elektrizität', 'feuer', 'fernzuhalten', 'übertreibt börse', 'foundation', 'fakten erfinden', 'fakten', 'fachleute einführung', 'fachleute', 'experimente gemini']


In [10]:
tfidf_words = calc_tf_idf(sent_dict.values(), stopwords)
with open(r"generated_data/Leipzig_sentences_tfidf.txt", "w") as f:
        f.write(str(tfidf_words))

['jakob hummel', 'obergrochlitz greiz', 'zweiradmechatroniker', 'greiz handwerk', 'zweiradmechatroniker tischler', 'tischler entschied', 'tischler', 'entschied jakob', 'obergrochlitz', 'hummel obergrochlitz', 'hummel', 'jakob', 'entschied', 'greiz', 'handwerk', 'garant lebendiger', 'garantie', 'garantie gewährleistung', 'garant', 'garantie sicherheit']


In [11]:
# TODO: tf-idf means per category?? which categories, ai terms?
''' 
vectorizer = TfidfVectorizer(lowercase= True)
means = {}
for i,doc in enumerate(corpora_lex.values()):
    if doc:
        X = vectorizer.fit_transform(doc).todense()
        # calc means for docs
        mean = np.mean(X)
        # to dict
        means[list(corpora_lex.keys())[i]] = mean
        mean_sorted = sorted(means.items(), key = lambda x : x[1], reverse = True)
pprint(mean_sorted)
'''

' \nvectorizer = TfidfVectorizer(lowercase= True)\nmeans = {}\nfor i,doc in enumerate(corpora_lex.values()):\n    if doc:\n        X = vectorizer.fit_transform(doc).todense()\n        # calc means for docs\n        mean = np.mean(X)\n        # to dict\n        means[list(corpora_lex.keys())[i]] = mean\n        mean_sorted = sorted(means.items(), key = lambda x : x[1], reverse = True)\npprint(mean_sorted)\n'